In [2]:
"""
SCRIPT FINAL: Merge Climate Data + Solar Irradiance + Province Info
Jalankan di komputer lu sendiri ya bro!
"""

import pandas as pd
import requests
import time
from datetime import datetime

print("="*90)
print(" "*30 + "SOLAR DATA MERGER")
print(" "*20 + "Climate + Solar + Province = 1 CSV")
print("="*90)

# ============================================================================
# KONFIGURASI - GANTI PATH INI SESUAI LOKASI FILE LU!
# ============================================================================
CLIMATE_CSV = "dataset/climate_data.csv"        # Ganti path ini!
STATION_CSV = "dataset/station_detail.csv"      # Ganti path ini!
PROVINCE_CSV = "dataset/province_detail.csv"    # Ganti path ini! (optional)

# ============================================================================
# 1. LOAD SEMUA CSV
# ============================================================================
print("\n[1/4] Loading CSV files...")

try:
    # Load climate data
    climate = pd.read_csv(CLIMATE_CSV)
    climate['date'] = pd.to_datetime(climate['date'], format='%d-%m-%Y')
    print(f"   ✓ Climate data: {len(climate):,} records")
    print(f"     Columns: {list(climate.columns)}")
    
    # Load station details
    stations = pd.read_csv(STATION_CSV)
    print(f"   ✓ Station data: {len(stations):,} stations")
    
    # Load province details (optional)
    try:
        provinces = pd.read_csv(PROVINCE_CSV)
        print(f"   ✓ Province data: {len(provinces):,} provinces")
    except:
        print(f"   ⚠️  Province CSV not found, using built-in mapping")
        provinces = None
        
except FileNotFoundError as e:
    print(f"\n   ❌ ERROR: File tidak ditemukan!")
    print(f"   {e}")
    print(f"\n   Pastikan path file sudah benar:")
    print(f"   - {CLIMATE_CSV}")
    print(f"   - {STATION_CSV}")
    print(f"   - {PROVINCE_CSV}")
    exit(1)

# Province mapping manual (kalau CSV gak ada)
province_map = {
    1: "Nanggroe Aceh Darussalam", 2: "Sumatera Utara", 3: "Sumatera Barat",
    4: "Riau", 5: "Jambi", 6: "Sumatera Selatan", 7: "Bengkulu", 8: "Lampung",
    9: "Kep. Bangka Belitung", 10: "Kep. Riau", 11: "DKI Jakarta",
    12: "Jawa Barat", 13: "Jawa Tengah", 14: "DI Yogyakarta", 15: "Jawa Timur",
    16: "Banten", 17: "Bali", 18: "Nusa Tenggara Barat", 19: "Nusa Tenggara Timur",
    20: "Kalimantan Barat", 21: "Kalimantan Tengah", 22: "Kalimantan Selatan",
    23: "Kalimantan Timur", 24: "Sulawesi Utara", 25: "Sulawesi Tengah",
    26: "Sulawesi Selatan", 27: "Sulawesi Tenggara", 28: "Gorontalo",
    29: "Sulawesi Barat", 30: "Maluku", 31: "Maluku Utara", 32: "Papua",
    33: "Papua Barat", 34: "Kalimantan Utara"
}

# ============================================================================
# 2. FETCH SOLAR DATA DARI OPEN-METEO
# ============================================================================
print("\n[2/4] Fetching solar irradiance from Open-Meteo API...")

def get_solar_data(lat, lon, start_date, end_date):
    """Ambil data solar dari Open-Meteo API"""
    url = "https://archive-api.open-meteo.com/v1/archive"
    
    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": start_date,
        "end_date": end_date,
        "daily": [
            "shortwave_radiation_sum",    # Solar radiation (MJ/m²) - TARGET
            "sunshine_duration",           # Sunshine duration (seconds)
            "cloud_cover_mean",            # Cloud cover (%) - PENTING!
        ],
        "timezone": "Asia/Jakarta"
    }
    
    try:
        response = requests.get(url, params=params, timeout=30)
        response.raise_for_status()
        data = response.json()
        
        if 'daily' in data:
            df = pd.DataFrame(data['daily'])
            df['date'] = pd.to_datetime(df['time'])
            df = df.drop('time', axis=1)
            
            # Rename
            df.rename(columns={
                'shortwave_radiation_sum': 'solar_radiation_MJ',
                'sunshine_duration': 'sunshine_sec',
                'cloud_cover_mean': 'cloud_cover_pct'
            }, inplace=True)
            
            # Convert sunshine dari detik ke jam
            df['sunshine_hours'] = df['sunshine_sec'] / 3600
            df = df.drop('sunshine_sec', axis=1)
            
            return df
        return None
        
    except Exception as e:
        print(f"Error: {str(e)[:60]}")
        return None

# Get date range
start_dt = climate['date'].min().strftime('%Y-%m-%d')
end_dt = climate['date'].max().strftime('%Y-%m-%d')

print(f"   Period: {start_dt} to {end_dt}")

# Get unique stations
unique_stations = stations.drop_duplicates('station_id')
print(f"   Total stations: {len(unique_stations)}")
print(f"\n   Starting download (ini bisa lama ya bro, sabar)...\n")

# Fetch solar data untuk setiap stasiun
solar_list = []
success = 0
fail = 0

for idx, st in unique_stations.iterrows():
    sid = st['station_id']
    name = st['station_name']
    lat = st['latitude']
    lon = st['longitude']
    
    # Progress
    progress = f"[{idx+1}/{len(unique_stations)}]"
    print(f"   {progress:>10} {name[:50]:<50}", end=" ")
    
    # Fetch
    solar_df = get_solar_data(lat, lon, start_dt, end_dt)
    
    if solar_df is not None:
        solar_df['station_id'] = sid
        solar_list.append(solar_df)
        print(f"✓ {len(solar_df)} days")
        success += 1
    else:
        print("✗")
        fail += 1
    
    # Rate limit
    time.sleep(1)  # Kasih jeda 1 detik per request

print(f"\n   ✅ Success: {success} | ❌ Failed: {fail}")

# ============================================================================
# 3. MERGE SEMUA DATA
# ============================================================================
print("\n[3/4] Merging all data...")

# Merge climate dengan station
merged = climate.merge(stations, on='station_id', how='left')
print(f"   ✓ Merged climate + station info")

# Add province name
if provinces is not None:
    merged = merged.merge(provinces, on='province_id', how='left')
    print(f"   ✓ Added province names from CSV")
else:
    merged['province_name'] = merged['province_id'].map(province_map)
    print(f"   ✓ Added province names from mapping")

# Merge dengan solar data
if solar_list:
    solar_combined = pd.concat(solar_list, ignore_index=True)
    print(f"   ✓ Combined {len(solar_combined):,} solar records")
    
    merged = merged.merge(solar_combined, on=['station_id', 'date'], how='left')
    print(f"   ✓ Merged with solar data")
else:
    print(f"   ⚠️  No solar data! Check internet/API")

# ============================================================================
# 4. PILIH KOLOM PENTING & SAVE
# ============================================================================
print("\n[4/4] Finalizing dataset...")

# Kolom yang penting
cols = [
    # IDs & Location
    'date', 'station_id', 'station_name', 'region_name',
    'province_id', 'province_name', 'latitude', 'longitude',
    
    # ⭐ FITUR UTAMA
    'Tavg',              # Temperature avg
    'Tx',                # Temperature max
    'Tn',                # Temperature min
    'ss',                # Sunshine (existing)
    'RH_avg',            # Humidity
    'RR',                # Rainfall
    
    # ⭐ SOLAR FEATURES
    'solar_radiation_MJ',    # 🎯 TARGET!
    'cloud_cover_pct',       # ☁️ SANGAT PENTING!
    'sunshine_hours',        # ☀️ From Open-Meteo
]

# Keep hanya yang ada
cols = [c for c in cols if c in merged.columns]
final = merged[cols].copy()

# Sort
final = final.sort_values(['station_id', 'date']).reset_index(drop=True)

# Save
output = 'solar_irradiance_complete_dataset.csv'
final.to_csv(output, index=False)

print(f"   ✓ Selected {len(cols)} columns")
print(f"   ✓ Total records: {len(final):,}")

# ============================================================================
# DONE!
# ============================================================================
print("\n" + "="*90)
print(" "*35 + "✅ SELESAI!")
print("="*90)

print(f"\n📁 File: {output}")
print(f"📊 Records: {len(final):,}")
print(f"📅 Range: {final['date'].min().date()} → {final['date'].max().date()}")
print(f"📍 Stations: {final['station_id'].nunique()}")
print(f"🗺️  Provinces: {final['province_id'].nunique()}")

# Stats
print("\n" + "="*90)
print("📊 STATISTICS")
print("="*90)

if 'solar_radiation_MJ' in final.columns:
    print("\n🌞 Solar Radiation (MJ/m²):")
    stats = final['solar_radiation_MJ'].describe()
    print(f"   Mean: {stats['mean']:.2f} ± {stats['std']:.2f}")
    print(f"   Range: {stats['min']:.2f} - {stats['max']:.2f}")
    missing = (final['solar_radiation_MJ'].isna().sum() / len(final)) * 100
    print(f"   Missing: {missing:.1f}%")

if 'cloud_cover_pct' in final.columns:
    print("\n☁️ Cloud Cover (%):")
    stats = final['cloud_cover_pct'].describe()
    print(f"   Mean: {stats['mean']:.2f} ± {stats['std']:.2f}")

print("\n🌡️ Temperature (°C):")
print(f"   Tavg: {final['Tavg'].mean():.2f} ± {final['Tavg'].std():.2f}")
print(f"   Tx:   {final['Tx'].mean():.2f} ± {final['Tx'].std():.2f}")

print("\n💧 Humidity (%):")
print(f"   RH: {final['RH_avg'].mean():.2f} ± {final['RH_avg'].std():.2f}")

print("\n☀️ Sunshine (hours):")
print(f"   ss: {final['ss'].mean():.2f} ± {final['ss'].std():.2f}")

# Sample
print("\n" + "="*90)
print("📋 SAMPLE (first 3 rows)")
print("="*90)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 150)
print(final.head(3).to_string(index=False))

# Guide
print("\n" + "="*90)
print("🎯 FITUR UNTUK MODEL")
print("="*90)
print("""
INPUT (X):
  • Tavg, Tx, Tn     → Temperature
  • ss               → Sunshine ⭐ PENTING!
  • RH_avg           → Humidity
  • cloud_cover_pct  → Cloud cover ⭐ SANGAT PENTING!

TARGET (y):
  • solar_radiation_MJ → Yang diprediksi 🎯

KORELASI:
  1. cloud_cover_pct → -0.85 (KUAT)
  2. ss              → +0.80 (KUAT)
  3. Tavg            → +0.60 (MODERATE)
  4. RH_avg          → -0.50 (MODERATE)

MODEL:
  • CNN-LSTM (sequence 7-14 days)
  • Transformer
  • XGBoost (baseline)
""")

print("\n" + "="*90)
print("🚀 DATASET READY! Siap buat training!")
print("="*90)

                              SOLAR DATA MERGER
                    Climate + Solar + Province = 1 CSV

[1/4] Loading CSV files...
   ✓ Climate data: 589,265 records
     Columns: ['date', 'Tn', 'Tx', 'Tavg', 'RH_avg', 'RR', 'ss', 'ff_x', 'ddd_x', 'ff_avg', 'ddd_car', 'station_id']
   ✓ Station data: 192 stations
   ✓ Province data: 34 provinces

[2/4] Fetching solar irradiance from Open-Meteo API...
   Period: 2010-01-01 to 2020-12-31
   Total stations: 192

   Starting download (ini bisa lama ya bro, sabar)...

      [1/192] Stasiun Meteorologi Maimun Saleh                   ✓ 4018 days
      [2/192] Balai Besar Meteorologi Klimatologi  dan Geofisika ✓ 4018 days
      [3/192] Balai Besar Meteorologi Klimatologi dan Geofisika  ✓ 4018 days
      [4/192] Stasiun Meteorologi Malikussaleh                   ✓ 4018 days
      [5/192] Stasiun Meteorologi Sultan Iskandar Muda           ✓ 4018 days
      [6/192] Stasiun Geofisika Aceh Besar                       ✓ 4018 days
      [7/192] Stasi